# Cell 1 - install dependencies

In [1]:
!pip install tensorflow numpy matplotlib

# Cell 2 — Import libraries

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Cell 3 — Generate training data

In [3]:
SAMPLES = 1000

x = np.random.uniform(low=0, high=2 * np.pi, size=SAMPLES)
y = np.sin(x)

# Add noise
y += 0.1 * np.random.randn(*y.shape)

# Split data
split = int(0.8 * SAMPLES)
x_train, x_test = x[:split], x[split:]
y_train, y_test = y[:split], y[split:]

print(f"Training samples: {len(x_train)}")
print(f"Test samples: {len(x_test)}")

Training samples: 800
Test samples: 200


# Cell 4 — Build and train model

In [4]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(1,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.fit(x_train, y_train, epochs=500, validation_data=(x_test, y_test), verbose=0)
print("Training complete!")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training complete!


# Cell 5 — Convert to TensorFlow Lite

In [5]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('sine_model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"Model size: {len(tflite_model)} bytes")

Saved artifact at '/tmp/tmpcj_q94w8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138077573362320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138077573363472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138077573362128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138077573360976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138077573364048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138077573358480: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model size: 3164 bytes


# Cell 6 — Convert to C array

In [6]:
!xxd -i sine_model.tflite > sine_model.h

# Fix variable name
with open('sine_model.h', 'r') as f:
    content = f.read()

content = content.replace('unsigned char sine_model_tflite[]', 'const unsigned char sine_model[]')
content = content.replace('unsigned int sine_model_tflite_len', 'const unsigned int sine_model_len')

with open('sine_model.h', 'w') as f:
    f.write(content)

print("sine_model.h generated!")

sine_model.h generated!


# Cell 7 — Download file

In [7]:
from google.colab import files
files.download('sine_model.h')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>